In [144]:
import openai
from pinecone import Pinecone, ServerlessSpec          # v3 client
from langchain_pinecone import PineconeVectorStore     # new LC wrapper
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.docstore.document import Document
from dotenv import load_dotenv
import os

load_dotenv()
# set API key
OPENAI_KEY=os.getenv("OPENAI_API_KEY")
OPENAI_BASE=os.getenv("OPENAI_API_BASE")
PINECONE_KEY = os.getenv('PINECONE_API_KEY')


openai_client = openai.OpenAI(
    base_url = OPENAI_BASE,
    api_key=OPENAI_KEY)


In [ ]:
# create vector DB and upsert docs
# create Pinecone vector db
INDEX_NAME       = "student"
DIMENSION        = 1536                    # 1536 for OpenAI text-embedding-3-small
METRIC           = "cosine"                # or "dotproduct", "euclidean"

# init pinecone 
pc = Pinecone(api_key=PINECONE_KEY)
# create index  
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(cloud="gcp", region="us-central1")
    )

index = pc.Index(INDEX_NAME)

# # Initialize your OpenAI embedder
embedder = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_KEY
)
# create the vector store 
vector_store = PineconeVectorStore(index, embedder)

# build and upload docs 
def build_docs(rows):
    docs=[]
    for kaid, turn_id, skill_tag, dt, fpm_state, text in rows:
        metadata={'kaid': kaid, 'turn_id': turn_id, 'skill_tag': skill_tag, 'date': str(dt), 'fpm_state': fpm_state}
        docs.append(Document(page_content=text, metadata=metadata))
    return docs

rows=client.query(query)
docs = build_docs(rows)

vector_store.add_documents(docs)

        

In [147]:
# create the memory block  
def build_memory_block(student_msg, kaid, k=4, skill_tag="regrouping-whole-numbers"):

  vec_results = vector_store.similarity_search(student_msg, 
                                               k=k, 
                                               filter=({"kaid" : kaid, "skill_tag": skill_tag})
                                               )
  memory_chunks = []
  for doc in vec_results:
      m = doc.metadata
      turns   = m.get("turn_id")
      skills  = m.get("skill_tag", [])
      dt = m.get("date", "")

      header = f"[turn {turns} | skills: {skills} | {dt}]"
      chunk  = f"{header}\n{doc.page_content}"
      memory_chunks.append(chunk)
      
  memory_block = "\n\n".join(memory_chunks)          # <- single string

  
  skill_state = {}
  for doc in vec_results:
    m = doc.metadata
    skill  = m.get("skill_tag", [])
    skill_level = m.get("fpm_state", "")
    skill_state[skill] = skill_level

  skill_lines = [f"{skill}: {level}" for skill, level in skill_state.items()]
  skill_block = "\n".join(skill_lines)  
  return memory_block, skill_block
  

In [148]:
student_msg = 'How can I regroup 52,340 using different place values'

system_prompt = ("""
# ROLE 
"You are an AI math tutor.
# YOUR PEDAGOGY RULES
1. Use Socratic questioning.
2. Encourage student to explain their reasoning aloud.
3. Do not do the work for the student and do not give away answers. 
# ADDITIONAL CONTEXT
You will be provided with two items: 
    1 - the history of past conversations between tutor and the student on this topic and 
    2 - student skill level on the given skill that you are tutoring. 
Use this information to adjust your tutoring approach as follows:
1. State students proficiency level on the skill.
2. If you see in the conversation history that the student already received help on this question, \
remind them that you already discussed this and ask if they remember that.  
3. If student is proficient on a skill acknowledge that and offer minimal support, like a broad strategy hint.
4. If a student is familiar or attempted on a skill, provide detailed support step by step

Here is an example:
    Student: How do I find common denominators?
    Tutor: I see that you are familiar with finding common denominators. ALso I see you asked about this before. Do you remember our previous discussion?                 
""")


# test memory input for tutor 
def tutor(student_msg, kaid,  model="gpt-4-khan", k=4, temperature=0.0):
    # get memory block
    memory_block, skill_set=build_memory_block(student_msg, kaid, k=k)
    
        
    completion = openai_client.chat.completions.create(
        model=model,
        temperature=0.0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "system", "content": 'the current knowldfge state on the relevant skill: ' + skill_set },
            {"role": "system", "content": 'relevant past conversations :'  + memory_block },
            
            {"role": "user", "content": student_msg}
            ]) 
    return completion.choices[0].message.content
    

tutor(student_msg, 'kaid_248328473686127753888179', k=4)

'I see that you are proficient in regrouping whole numbers. Also, I noticed that we have discussed this exact question before. Do you remember our previous discussion about regrouping the number 52,340 into different place values?'

In [ ]:
# convo simulator to test next and need to add a classifier to student question to detect skill

# def tutor(question, student_response_prompt=student_response_prompt, model="gpt-4-khan", system=None):
#     counter = 0
#     tutor_responses=[]
#     student_questions =[question]
#     while counter<3:
#         if counter==0:
#             completion = openai_client.chat.completions.create(
#                 model=model,
#                 temperature=0.0,
#                 messages=[
#                     {"role": "system", "content": system_prompt},
#                     {"role": "user", "content": question}
#                     ]) # ask first question
#             # store tutor response
#             tutor_responses.append(completion.choices[0].message.content)
            
#         else:
#             # generate next student question
#             student_q = student(tutor_responses[-1], student_response_prompt)
#             student_questions.append(student_q)
#             completion = openai_client.chat.completions.create(
#                 model=model,
#                 temperature=0.0,
#                 messages=[{"role": "user", "content": student_q}])
#             # generate tutor response to next question
#             tutor_responses.append(completion.choices[0].message.content)
                       
#         counter += 1
#     return pd.DataFrame({'question': student_questions, 'responses': tutor_responses})

# def student(response, student_response_prompt,  model="gpt-4-khan", system=None):
#     responses=[]
#     completion = openai_client.chat.completions.create(
#         model=model,
#         temperature=0.0,
#         messages=[{"role": "user", "content": student_response_prompt + response}])
#     response = completion.choices[0].message.content
#     responses.append(response)
#     return responses[-1]

# tutor(question)
